<a href="https://colab.research.google.com/github/mahnazsaiyed/prompt-engineering-assignment-9-23/blob/main/Exercise1_Prompt_Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1: Prompt Chaining for a Customer Support AI

## Goal
Build a simple multi-step prompt chain that simulates a customer service flow.

The prompt chain will:
1. Classify the customer's issue.
2. Determine what information is missing.
3. Generate an appropriate customer support response.
4. Decide whether the issue should be escalated.

Each step uses the output from the previous step as input for the next step.

## Tools Used

- Google Colab
- Python
- Google Gemini API

## Customer Scenario

Customer message:

"I ordered a pair of headphones five days ago and paid for express shipping, but my order still hasn't arrived. The tracking page hasn't updated in three days."

## Step 1: Classify the Customer Issue

The first prompt analyzes the customer's message and classifies the issue into a customer support category. The output of this step will be passed to Step 2.

The prompt includes constraints so that the AI returns a clear and consistent response.

### Prompt 1

You are a customer support assistant.

Analyze the customer's message and classify the issue into ONE of these categories:
- Shipping/Delivery
- Billing/Payment
- Returns/Refunds
- Product Issue
- Account Issue
- Other

Return only the category name and a one-sentence explanation.

Customer message:
"I ordered a pair of headphones five days ago and paid for express shipping, but my order still hasn't arrived. The tracking page hasn't updated in three days."

In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

print("Gemini API connected successfully!")

Gemini API connected successfully!


In [ ]:
customer_message = """
I ordered a pair of headphones five days ago and paid for express shipping,
but my order still hasn't arrived. The tracking page hasn't updated in three days.
"""

print(customer_message)


I ordered a pair of headphones five days ago and paid for express shipping,
but my order still hasn't arrived. The tracking page hasn't updated in three days.



In [ ]:
prompt1 = f"""
You are a customer support assistant.

Analyze the customer's message and classify the issue into ONE of these categories:
- Shipping/Delivery
- Billing/Payment
- Returns/Refunds
- Product Issue
- Account Issue
- Other

Return only the category name and a one-sentence explanation.

Customer message:
{customer_message}
"""

response1 = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt1
)

classification = response1.text

print("STEP 1 - ISSUE CLASSIFICATION")
print(classification)

STEP 1 - ISSUE CLASSIFICATION
**Shipping/Delivery**

The customer is concerned about a delayed order and a lack of tracking updates for a package that was supposed to arrive via express shipping.


### Testing and Iteration

During testing, the original code used the `gemini-2.5-flash` model. When executed, the API returned a 404 error because this model was no longer available to new users. I updated the model to `gemini-3.6-flash` and reran the prompt successfully.

## Step 2: Identify Missing Information

This prompt uses the issue classification from Step 1 along with the original customer message. It identifies information that customer support would need before recommending a solution.

In [ ]:
prompt2 = f"""
You are a customer support assistant.

Using the original customer message AND the classification from Step 1,
identify what information is still needed to help resolve the customer's issue.

Ask for no more than 3 pieces of missing information.
Do not ask for sensitive information such as passwords or payment card numbers.
Format the response as a numbered list.

Original customer message:
{customer_message}

Classification from Step 1:
{classification}
"""

response2 = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt2
)

missing_info = response2.text

print("STEP 2 - MISSING INFORMATION")
print(missing_info)

STEP 2 - MISSING INFORMATION
1. Your order number
2. The email address or phone number used at checkout


## Step 3: Propose a Customer Support Solution

This prompt uses the original customer message, the classification from Step 1, and the missing information identified in Step 2 to create an appropriate customer support response.

In [ ]:
prompt3 = f"""
You are a professional customer support representative.

Create a helpful response to the customer using the information from the
previous steps.

Requirements:
- Acknowledge the customer's frustration.
- Clearly explain what information is needed next.
- Do not promise a refund, replacement, or delivery date without verification.
- Use a polite and professional tone.
- Keep the response under 120 words.

Original customer message:
{customer_message}

Issue classification from Step 1:
{classification}

Missing information from Step 2:
{missing_info}
"""

response3 = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt3
)

solution = response3.text

print("STEP 3 - PROPOSED CUSTOMER RESPONSE")
print(solution)

STEP 3 - PROPOSED CUSTOMER RESPONSE
Hello,

I completely understand your frustration regarding the delay with your headphones, especially since you paid for express shipping and the tracking hasn't updated. I would be concerned too, and I apologize for the worry this has caused.

To help you look into this right away and check the status of your shipment, could you please provide:

1. Your order number
2. The email address or phone number used at checkout

Once we have this information, our team will investigate the delivery status with the carrier right away. Thank you for your patience, and I look forward to helping you resolve this.

Best regards,

Customer Support Team


## Step 4: Determine Whether Escalation Is Needed

This final prompt uses the outputs from the previous steps to determine whether the customer's issue should be escalated to a human support agent.

In [ ]:
prompt4 = f"""
You are a customer support escalation assistant.

Review the customer's original issue and the outputs from the previous
steps. Decide whether the issue should be escalated to a human support agent.

Escalate if:
- The issue cannot be resolved without accessing private order records.
- A refund, replacement, or payment adjustment may be required.
- The delivery problem requires investigation by a human support agent.

Return exactly:
Decision: ESCALATE or DO NOT ESCALATE
Reason: One sentence explaining the decision.

Original customer message:
{customer_message}

Classification from Step 1:
{classification}

Missing information from Step 2:
{missing_info}

Proposed response from Step 3:
{solution}
"""

response4 = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt4
)

escalation = response4.text

print("STEP 4 - ESCALATION DECISION")
print(escalation)

STEP 4 - ESCALATION DECISION
Decision: ESCALATE
Reason: The delivery problem with the un-updated tracking for an express shipment requires investigation by a human support agent once the customer provides their details.


## Testing and Iteration

During testing, I initially used the `gemini-2.5-flash` model. When I ran the code, the API returned a 404 error because the model was no longer available to new users. I updated the model and successfully generated the Step 1 classification.

During Step 2, I received a 503 error because the model was experiencing high demand. I tested the prompt again and then changed the model to `gemini-3.5-flash-lite`. After this change, the remaining prompts ran successfully.

This testing process helped me identify API/model availability issues and adjust the code while keeping the prompt chain logic the same.

## Final Result

The prompt chain successfully processed the customer support issue through four connected stages:

1. Classified the issue as Shipping/Delivery.
2. Identified missing customer information.
3. Generated a professional customer support response.
4. Determined that the issue should be escalated to a human support agent.

Each stage used information generated by an earlier stage, creating a connected multi-step customer support workflow.